In [1]:
import pandas as pd
import os
import glob

# Define the folder and get files
folder = "../../data/kinect_good_preprocessed_A9_mediapipe"
all_files = glob.glob(os.path.join(folder, "*.csv"))

# Define Left/Right pairs based on your columns
pairs = [
    ('left_shoulder', 'right_shoulder'),
    ('left_elbow', 'right_elbow'),
    ('left_wrist', 'right_wrist'),
    ('left_hip', 'right_hip'),
    ('left_knee', 'right_knee'),
    ('left_ankle', 'right_ankle')
]

for f in all_files:
    if "_mirrored" in f: continue
    
    df = pd.read_csv(f)
    mirrored_df = df.copy()
    
    # 1. Flip X-coordinates
    # We assume X is normalized 0-1. If it's pixel space (e.g. 640), change 1.0 to 640.
    x_cols = [c for c in df.columns if "_3d_x" in c]
    mirrored_df[x_cols] = 1.0 - df[x_cols]
    
    # 2. Swap the data between Left and Right columns
    for left, right in pairs:
        for axis in ['x', 'y', 'z']:
            l_col = f"{left}_3d_{axis}"
            r_col = f"{right}_3d_{axis}"
            
            # Swap values: Left becomes Right's original (but flipped X), and vice versa
            mirrored_df[l_col] = df[r_col]
            mirrored_df[r_col] = df[l_col]
            
            # CRITICAL: Because we swapped the columns, the X-flip we did in step 1 
            # is now on the wrong side. We must re-flip X for the swapped columns.
            if axis == 'x':
                mirrored_df[l_col] = 1.0 - mirrored_df[l_col]
                mirrored_df[r_col] = 1.0 - mirrored_df[r_col]

    # Save with a new suffix
    new_path = f.replace(".csv", "_mirrored.csv")
    mirrored_df.to_csv(new_path, index=False)

print(f"Successfully mirrored {len(all_files)} files.")

Successfully mirrored 179 files.
